# Bronze & Silver layer definitions

Schemas and transformation logic for the Bronze (raw) and Silver (cleansed) layers.
This notebook only defines things, it does not write any table. It is loaded with
`%run ./layers` by `pipeline_utils`, and the actual loading is done by `run_pipeline`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType,
    BooleanType, TimestampType
)

# Bronze schema: the record exactly as the API returns it.
# All leaf fields are strings on purpose. If the source changes a type
# (e.g. value comes as text instead of a number) Bronze still keeps the raw
# value, and the casting / validation happens on the way to Silver.
bronze_schema = StructType([
    StructField("indicator", StructType([
        StructField("id", StringType(), True),
        StructField("value", StringType(), True)
    ]), True),
    StructField("country", StructType([
        StructField("id", StringType(), True),
        StructField("value", StringType(), True)
    ]), True),
    StructField("countryiso3code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("value", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("obs_status", StringType(), True),
    StructField("decimal", StringType(), True)
])

RAW_FIELDS = [f.name for f in bronze_schema.fields]

# Silver schema (target model, the table is created from this in pipeline_utils)
# Primary key: (country_key, indicator_code, year)
silver_schema = StructType([
    StructField("country_key", StringType(), False),
    StructField("country_code", StringType(), True),
    StructField("country_iso3", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("indicator_code", StringType(), False),
    StructField("indicator_name", StringType(), True),
    StructField("year", IntegerType(), False),
    StructField("value", DoubleType(), True),
    StructField("is_reported", BooleanType(), True),
    StructField("decimal_places", IntegerType(), True),
    StructField("source_batch_id", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("source_load_timestamp", TimestampType(), True),
    StructField("load_timestamp", TimestampType(), False)
])

SILVER_COLUMNS = [f.name for f in silver_schema.fields]
SILVER_KEY = ["country_key", "indicator_code", "year"]

In [0]:
def read_raw_file(path):
    """Read one raw JSON file using the explicit Bronze schema (no schema inference).

    Fields that are not part of the schema (a new column added by the source) or
    that cannot be parsed into it are not lost, they are captured in _rescued_data.
    """
    df = (
        spark.read
        .schema(bronze_schema)
        .option("multiline", "true")
        .option("mode", "PERMISSIVE")
        .option("rescuedDataColumn", "_rescued_data")
        .json(path)
    )
    if "_rescued_data" not in df.columns:
        df = df.withColumn("_rescued_data", F.lit(None).cast(StringType()))
    return df


def add_bronze_metadata(df, source_file, batch_id, load_type, load_ts):
    """Add lineage columns. record_hash identifies the exact content of a raw record."""
    return (
        df
        .withColumn("record_hash", F.sha2(F.to_json(F.struct(*RAW_FIELDS)), 256))
        .withColumn("source_file", F.lit(source_file))
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("load_type", F.lit(load_type))
        .withColumn("load_timestamp", F.lit(load_ts).cast(TimestampType()))
    )


def _blank(c):
    return F.coalesce(F.trim(F.col(c)), F.lit("")) == ""


def bronze_reject_reason():
    """Records that cannot be identified at all are quarantined instead of loaded."""
    return (
        F.when(_blank("indicator.id"), F.lit("missing_indicator_id"))
        .when(_blank("date"), F.lit("missing_date"))
        .when(_blank("country.id") & _blank("country.value"), F.lit("missing_country"))
    )

In [0]:
def _clean(c):
    v = F.trim(F.col(c))
    return F.when(v == "", F.lit(None)).otherwise(v)


def to_silver(bronze_df, load_ts):
    """Flatten, clean and cast Bronze records into the Silver model.

    try_cast is used so that a bad value gives NULL (and is quarantined later)
    instead of failing the whole batch.
    """
    df = bronze_df.select(
        _clean("country.id").alias("country_code"),
        _clean("countryiso3code").alias("country_iso3"),
        _clean("country.value").alias("country_name"),
        _clean("indicator.id").alias("indicator_code"),
        _clean("indicator.value").alias("indicator_name"),
        F.col("date").alias("year_raw"),
        F.col("value").alias("value_raw"),
        F.col("decimal").alias("decimal_raw"),
        F.col("record_hash"),
        F.col("batch_id").alias("source_batch_id"),
        F.col("source_file"),
        F.col("load_timestamp").alias("source_load_timestamp"),
    )
    return (
        df
        # Some entities (e.g. Niue, Tokelau, Anguilla) come with an empty country id
        # and empty ISO3 code, so the name is used as the key for those rows.
        .withColumn(
            "country_key",
            F.coalesce(
                F.col("country_code"),
                F.concat(F.lit("NAME_"), F.upper(F.regexp_replace(F.col("country_name"), "[^A-Za-z0-9]+", "_")))
            )
        )
        .withColumn("year", F.expr("try_cast(trim(year_raw) AS INT)"))
        .withColumn("value", F.expr("try_cast(trim(value_raw) AS DOUBLE)"))
        .withColumn("decimal_places", F.expr("try_cast(trim(decimal_raw) AS INT)"))
        .withColumn("is_reported", F.col("value").isNotNull())
        .withColumn("load_timestamp", F.lit(load_ts).cast(TimestampType()))
    )


def silver_reject_reason():
    """Rows that fail casting are quarantined, they never reach the Silver table."""
    return (
        F.when(F.col("year").isNull() | ~F.col("year").between(1900, 2100), F.lit("invalid_year"))
        .when(
            F.col("value_raw").isNotNull() & (F.trim(F.col("value_raw")) != "") & F.col("value").isNull(),
            F.lit("value_not_numeric")
        )
    )


def latest_per_key(df):
    """Keep one row per Silver key: the most recently loaded one (deterministic order)."""
    w = Window.partitionBy(*SILVER_KEY).orderBy(
        F.col("source_load_timestamp").desc(),
        F.col("source_batch_id").desc(),
        F.col("source_file").desc(),
        F.col("record_hash")
    )
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")